# 01 park

In [5]:
import geopandas as gpd

gdf = gpd.read_file(r"D:\shanghai workshop\building+park+waterbody\landuse-公园用地\landuse_park.shp")
print(gdf.crs)


EPSG:4326


In [8]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin

def polygon_to_binary_raster_projected(shapefile_path, out_raster_path, resolution=0.15):
    gdf = gpd.read_file(shapefile_path)

    # 若为经纬度坐标系，则投影为 UTM 51N（单位：米）
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)

    # 获取边界
    minx, miny, maxx, maxy = gdf.total_bounds
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)
    transform = from_origin(minx, maxy, resolution, resolution)

    # 构造 rasterize 目标
    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty]

    raster = rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8"
    )

    # 写入为 GeoTIFF
    with rasterio.open(
        out_raster_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs=gdf.crs,  # 已转为米单位
        transform=transform
    ) as dst:
        dst.write(raster, 1)


polygon_to_binary_raster_projected(
    shapefile_path=r"D:\shanghai workshop\building+park+waterbody\landuse-公园用地\landuse_park.shp",
    out_raster_path=r"D:\shanghai workshop\raster\park.tif",
    resolution=30 # 30m
)

# 02 water

In [9]:
import geopandas as gpd

gdf = gpd.read_file(r"D:\BaiduNetdiskDownload\3-shanghai-latest-free.shp\gis_osm_water_a_free_1.shp")
print(gdf.crs)

EPSG:4326


In [10]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin

def polygon_to_categorical_raster(shapefile_path, out_raster_path, field_name="fclass", resolution=30):
    gdf = gpd.read_file(shapefile_path)

    # 如果不是投影坐标系，转换为 UTM 51N（单位：米）
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)

    # 获取唯一类别并编码为整数
    unique_classes = gdf[field_name].dropna().unique()
    class_to_id = {cls: idx for idx, cls in enumerate(sorted(unique_classes))}
    print("类别编码映射：", class_to_id)

    # 创建 (geometry, value) 对
    shapes = [
        (geom, class_to_id.get(val, 0))  # 找不到类别时默认值为 0
        for geom, val in zip(gdf.geometry, gdf[field_name])
        if geom is not None and not geom.is_empty and val in class_to_id
    ]

    # 计算边界和分辨率
    minx, miny, maxx, maxy = gdf.total_bounds
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)
    transform = from_origin(minx, maxy, resolution, resolution)

    # 栅格化
    raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8"
    )

    # 保存为 GeoTIFF
    with rasterio.open(
        out_raster_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs=gdf.crs,
        transform=transform
    ) as dst:
        dst.write(raster, 1)

    # 返回类别映射（可选）
    return class_to_id

# 示例调用
class_map = polygon_to_categorical_raster(
    shapefile_path=r"D:\BaiduNetdiskDownload\3-shanghai-latest-free.shp\gis_osm_water_a_free_1.shp",
    out_raster_path=r"D:\shanghai workshop\raster\water.tif",
    field_name="fclass",
    resolution=30
)


类别编码映射： {'dock': 0, 'reservoir': 1, 'riverbank': 2, 'water': 3, 'wetland': 4}
